In [ ]:
# Clone the repository and move to the stable branch
!git clone https://github.com/aryonmt/clickbait-spoiling-IR-final-project.git
%cd clickbait-spoiling-IR-final-project
!git checkout phase1-reproducibility

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import glob

# Ensure data directory exists in the active workspace
os.makedirs("data", exist_ok=True)

# Scan all Kaggle input folders for the SemEval data assets
train_paths = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
val_paths = glob.glob("/kaggle/input/**/validation.jsonl", recursive=True)

if train_paths:
    print(f"Target train dataset found: {train_paths[0]}")
    if not os.path.exists("data/train.jsonl"):
        os.symlink(train_paths[0], "data/train.jsonl")
        print("Created symbolic link to data/train.jsonl")
else:
    print("[WARNING] Could not find 'train.jsonl' in Kaggle inputs.")

if val_paths:
    print(f"Target validation dataset found: {val_paths[0]}")
    if not os.path.exists("data/validation.jsonl"):
        os.symlink(val_paths[0], "data/validation.jsonl")
        print("Created symbolic link to data/validation.jsonl")
else:
    print("[WARNING] Could not find 'validation.jsonl' in Kaggle inputs.")

In [ ]:
!python -m main_transformer_task1 \
    --train_path data/train.jsonl \
    --val_path data/validation.jsonl \
    --model_name roberta-base \
    --output_dir results_task1_roberta \
    --lr 2e-5

In [ ]:
!python -m main_transformer_task2_qa \
    --train_path data/train.jsonl \
    --val_path data/validation.jsonl \
    --model_name deepset/roberta-base-squad2 \
    --output_dir results_qa \
    --lr 3e-5

In [ ]:
!python -m main_transformer_task1 \
    --train_path data/train.jsonl \
    --val_path data/validation.jsonl \
    --model_name microsoft/deberta-v3-base \
    --output_dir results_task1_deberta \
    --lr 1e-5

In [ ]:
!python -m main_integrated_pipeline \
    --val_path data/validation.jsonl \
    --t1_checkpoint results_task1_roberta/checkpoint-300 \
    --t2_checkpoint clickbait-spoiling-IR-final-project/results_qa/checkpoint-1218 \
    --output_path run_integrated_pipeline.jsonl

In [ ]:
# Generate the detailed report folder
!python -m main_generate_report \
    --predictions_path run_integrated_pipeline.jsonl \
    --val_path data/validation.jsonl

# Archive the generated report plots and HTML into a single zip file for quick download
!zip -r reports.zip reports/